## Legal Expert Knowledge Worker

This is a question-answering assistant with access to the Constitution (supreme law), statutory law, English common law, customary law and Islamic law documents. This will make it an expert in legal problems and it will answer any legal questions thrown to it.

### Note: The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

### PART A: Retrieve the documents and Divide them into chunks

The documents will be retrieved from an external source (e.g. Google Drive).

In [ ]:
%pip install google-api-python-client google-auth google-auth-oauthlib
%pip install 'markitdown[all]'


In [ ]:
import os
from dotenv import load_dotenv

import tiktoken
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go


load_dotenv(override=True)

from google_drive_client import GoogleDriveClient

google_drive_credentials_path = os.getenv('GOOGLE_APPLICATION_CREDENTIALS')
google_drive_folder_id = os.getenv('GOOGLE_DRIVE_FOLDER_ID')

print(f"Credentials path: {google_drive_credentials_path}")
print(f"Google Drive Folder ID: {google_drive_folder_id}")

# Google Drive client:
collector = GoogleDriveClient(google_drive_credentials_path, google_drive_folder_id)
collector.collect_files()

# Files from a Google Drive folder in markdown format.
documents = collector.get_collection()

# Entire knowledge base
entire_knowledge_base = collector.entire_knowledge_base

print(f"Found {len(documents)} files in the knowledge base")

In [ ]:

from openai import OpenAI

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')


OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}


Check the number of tokens.

In [ ]:
# How many tokens in all the documents?
model = MODEL_MAP["GPT"]["model"]

def count_tokens(model, knowledge_base):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(knowledge_base)
    token_count = len(tokens)
    return token_count

token_count = count_tokens(model, entire_knowledge_base)

print(f"Total tokens for {model}: {token_count:,}")

# print(documents['The_Constitution_of_Kenya_2010.md'][0:1000])
# print(documents['companies-act-2015.md'][0:1000])


Divide the documents into chunks.

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter
# But first convert the documents to a list of langchain documents.
from langchain_core.documents import Document

langchain_documents = [
    Document(page_content=content, metadata={"file_name": filename})
    for filename, content in documents.items()
]

def split_documents(langchain_documents, chunk_size, chunk_overlap):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = text_splitter.split_documents(langchain_documents)
    return chunks

chunks = split_documents(langchain_documents, 1000, 200)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

chunks[100]

### PART B: Make vectors and store in Chroma

In [ ]:
# Pick an embedding model
db_name = "vector_db"
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
#embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

### Part C: Visualize!

This section visualizes the document embeddings so we can inspect how chunks cluster by source document.

- Fetches embeddings, documents, and metadata from the Chroma collection via `collection.get()`; converts embeddings to a NumPy array; derives `sources` from `file_name` metadata; assigns `colors` by document (red = Constitution, blue = Companies Act) 
- Applies **t-SNE** with `n_components=2` to reduce 384‑dim vectors to 2D; builds an interactive Plotly `go.Scatter` plot; hover shows source and text preview; layout 800×600 
- Same pipeline with **t-SNE** `n_components=3`; uses `go.Scatter3d` for a rotatable 3D scatter plot; layout 900×700 with x/y/z axes titled.


In [ ]:
# Prework
import numpy as np

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
sources = [metadata['file_name'] for metadata in metadatas]
colors = [['red', 'blue'][['The_Constitution_of_Kenya_2010.md', 'companies-act-2015.md'].index(t)] for t in sources]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(sources, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(sources, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

### Expert Question Answerer for Kenyan Company Law

LangChain 1.0 implementation of a RAG pipeline.
Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [ ]:
# Connect to Chroma; use Hugging Face all-MiniLM-L6-v2
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

### Set up the 2 key LangChain objects: retriever and llm


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
import gradio as gr

base_url = MODEL_MAP["GPT"]["endpoint"]
api_key = MODEL_MAP["GPT"]["api_key"]

MODEL = "gpt-4.1-nano"

retriever = vectorstore.as_retriever()
llm = ChatOpenAI(
    base_url=base_url,
    api_key=api_key,
    temperature=0, model_name=MODEL)

Retriever

In [ ]:
retriever.invoke("What is the financial year of a company?")

LLM

In [ ]:
llm.invoke("What is the financial year of a company?")

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a legal expert knowledgeable or conversant with Kenyan company law. 
You are chatting with a user about Kenyan company law.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.

Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("What does the law say about leadership in a company?", [])


In [ ]:
gr.ChatInterface(answer_question).launch()